# 19.3 Working with complementarity constraints

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

All of AMPL's features for ordinary equalities and inequalities extend in a straightforward
way to complementarity constraints. This section covers extensions in three areas:
expressions for related solution values, effects of presolve and related displays of problem
statistics, and generic synonyms for constraints.

**Related solution values**

AMPL's built-in suffixes for values related to a problem and its solution extend to
complementarity constraints, but with two collections of suffixes — of the form
cname.Lsuf and cname.Rsuf — corresponding to the left and right operands of complements,
respectively. Thus after econ2.mod ([Figure 19-4](./19_1_sources_of_complementarity.ipynb#fig-19-4)) has been solved, for
example, we can use the following display command to look at values associated with
the constraint Lev_Compl:

```ampl
ampl: display Lev_Compl.Llb, Lev_Compl.Lbody,
ampl?         Lev_Compl.Rbody, Lev_Compl.Rslack;
: Lev_Compl.Llb Lev_Compl.Lbody Lev_Compl.Rbody Lev_Compl.Rslack :=
P1       240           240      1392.86             Infinity
P1a      270          1000      -824.286            Infinity
P2       220           220       264.286            Infinity
P2a      260           680         5.00222e-12      Infinity
P2b      200           200       564.286            Infinity
P3       260           260       614.286            Infinity
P3c      220          1000      -801.429            Infinity
P4       240           240       584.286            Infinity
;
```

Because the right operand of Lev_Compl is an expression, it is treated as a "constraint"
with infinite lower and upper bounds, and hence infinite slack.

A suffix of the form cname.slack is also defined for complementarity constraints.
For complementary pairs of single inequalities, it is equal to the lesser of
cname.Lslack and cname.Rslack. Hence it is nonnegative if and only if both
inequalities are satisfied and is zero if the complementarity constraint holds exactly. For
complementary double inequalities of the form

```ampl
expr complements lbound <= body <= ubound
lbound <= body <= ubound complements expr
```

cname.slack is defined to be

```ampl
min(expr, body - lbound)               if body <= lbound
min(-expr, ubound - body)              if body >= ubound
-abs(expr)                             otherwise
```

Hence in this case it is always nonpositive, and is zero when the complementarity constraint
is satisfied exactly.

If cname for a complementarity constraint appears unsuffixed in an expression, it is
interpreted as representing cname.slack.

**Presolve**

As explained in [Section 14.1](../14/14_1_presolve.ipynb), AMPL incorporates a presolve phase that can substantially
simplify some linear programs. In the presence of complementarity constraints,
several new kinds of simplifications become possible.

As an example, given a constraint of the form

```ampl
expr 1 >= 0 complements expr 2 >= 0
```

if presolve can deduce that expr 1 is strictly positive for all feasible points — in other
words, that it has a positive lower bound — it can replace the constraint by expr 2 = 0.

Similarly, in a constraint of the form

```ampl
lbound <= body <= ubound complements expr
```

there are various possibilities, including the following:

```ampl
If presolve can deduce for all      Then the constraint can be replaced by
feasible points that
  body < ubound                      lbound <= body complements expr >= 0
  lbound < body < ubound             expr = 0
  expr < 0                           body = ubound
```

Transformations of these kinds are carried out automatically, unless option presolve 0
is used to turn off the presolve phase. As with ordinary constraints, results are reported
in terms of the original model.

By displaying a few predefined parameters:

```ampl
_ncons          the number of ordinary constraints before presolve
_nccons         the number of complementarity conditions before presolve
_sncons         the number of ordinary constraints after presolve
_snccons        the number of complementarity conditions after presolve
```

or by setting option show_stats 1, you can get some information on the number of
simplifying transformations that presolve has made:

```ampl
ampl:   model econ2.mod; data econ2.dat;
ampl:   option solver path;
ampl:   option show_stats 1;
ampl:   solve;
Presolve eliminates 16 constraints and 2 variables.

Presolve resolves 2 of 14 complementarity conditions.
Adjusted problem:
12 variables, all linear
12 constraints, all linear; 62 nonzeros
12 complementarity conditions among the constraints:
       12 linear, 0 nonlinear.
0 objectives.

Path v4.5: Solution found.

7 iterations (1 for crash); 30 pivots.
8 function, 8 gradient evaluations.

ampl: display _ncons, _nccons, _sncons, _snccons;
_ncons = 28
_nccons = 14
_sncons = 12
_snccons = 12
```

When first instantiating the problem, AMPL counts each complementarity constraint
as two ordinary constraints (the two arguments to complements) and also as a complementarity
condition. Thus \_nccons equals the number of complementarity constraints
before presolve, and \_ncons equals twice \_nccons plus the number of any noncomplementarity
constraints before presolve. The presolve messages at the beginning of
the show_stats output indicate how much presolve was able to reduce these numbers.

In this case the reason for the reduction can be seen by comparing each product's
demand to the minimum possible output of that product — the amount that results from
setting each Level[j] to level_min[j]:

```ampl
ampl: display {i in PROD}
ampl?    (sum{j in ACT} io[i,j]*level_min[j], demand[i]);
:   sum{j in ACT} io[i,j]*level_min[j] demand[i]    :=
AA1                 69820                 70000
AC1                 45800                 80000
BC1                 37300                 90000
BC2                 29700                 70000
NA2               1854920                 4e+05
NA3                843700                 8e+05
;
```

We see that for products NA2 and NA3, the total output exceeds demand even at the lowest
activity levels. Hence in the constraint

```ampl
subject to Pri_Compl {i in PROD}:
       Price[i] >= 0 complements
       sum {j in ACT} io[i,j] * Level[j] >= demand[i];
```

the right-hand argument to complements never holds with equality for NA2 or NA3.
Presolve thus concludes that Price["NA2"] and Price["NA3"] can be fixed at
zero, removing them from the resulting problem.
**Generic synonyms**

AMPL's generic synonyms for constraints ([Section 12.6](../12/12_6_accessing_model_and_solver_status.ipynb)) extend to complementarity
conditions, mainly through the substitution of ccon for con in the synonym names.

From the modeler's view (before presolve), the ordinary constraint synonyms remain:

```ampl
_ncons               number of ordinary constraints before presolve
_conname             names of the ordinary constraints before presolve
_con                 synonyms for the ordinary constraints before presolve
```

The complementarity constraint synonyms are:

```ampl
_nccons              number of complementarity constraints before presolve
_cconname            names of the complementarity constraints before presolve
_ccon                synonyms for the complementarity constraints before presolve
```

Because each complementarity constraint also gives rise to two ordinary constraints, as
explained in the preceding discussion of presolve, there are two entries in \_conname
corresponding to each entry in \_cconname:

```ampl
ampl: display {i in 1..6} (_conname[i], _cconname[i]);
:       _conname[i]           _cconname[i]       :=
1   "Pri_Compl['AA1'].L"   "Pri_Compl['AA1']"
2   "Pri_Compl['AA1'].R"   "Pri_Compl['AC1']"
3   "Pri_Compl['AC1'].L"   "Pri_Compl['BC1']"
4   "Pri_Compl['AC1'].R"   "Pri_Compl['BC2']"
5   "Pri_Compl['BC1'].L"   "Pri_Compl['NA2']"
6   "Pri_Compl['BC1'].R"   "Pri_Compl['NA3']"
;
```

For each complementarity constraint cname, the left and right arguments to the complements
operator are the ordinary constraints named cname.L and cname.R. This is confirmed
by using the synonym terminology to expand the complementarity constraint
Pri_Compl['AA1'] and the corresponding two ordinary constraints from the example
above:

```ampl
ampl: expand Pri_Compl['AA1'];
subject to Pri_Compl['AA1']:
        Price['AA1'] >= 0
   complements
        60*Level['P1'] + 8*Level['P1a'] + 8*Level['P2'] +
        40*Level['P2a'] + 15*Level['P2b'] + 70*Level['P3'] +
        25*Level['P3c'] + 60*Level['P4'] >= 70000;
ampl: expand _con[1], _con[2];
subject to Pri_Compl.L['AA1']:
        Price['AA1'] >= 0;
subject to Pri_Compl.R['AA1']:
        60*Level['P1'] + 8*Level['P1a'] + 8*Level['P2'] +
        40*Level['P2a'] + 15*Level['P2b'] + 70*Level['P3'] +
        25*Level['P3c'] + 60*Level['P4'] >= 70000;
```

From the solver's view (after presolve), a more limited collection of synonyms is defined:

```ampl
_sncons               number of all constraints after presolve
_snccons              number of complementarity constraints after presolve
_sconname             names of all constraints after presolve
_scon                 synonyms for all constraints after presolve
```

Necessarily \_snccons is less than or equal to \_sncons, with equality only when all
constraints are complementarity constraints.

To simplify the problem description that is sent to the solver, AMPL converts every
complementarity constraint into one of the following canonical forms:

```ampl
expr complements lbound <= var <= ubound
expr <= 0 complements var <= ubound
expr >= 0 complements lbound <= var
```

where `var` is the name of a different variable for each constraint. (Where an expression
more complicated than a single variable appears on both sides of complements, this
involves the introduction of an auxiliary variable and an equality constraint defining the
variable to equal one of the expressions.) By using solexpand in place of expand,
you can see the form in which AMPL has sent a complementarity constraint to the solver:

```ampl
ampl: solexpand Pri_Compl['AA1'];
subject to Pri_Compl['AA1']:
   -70000 + 60*Level['P1'] + 8*Level['P1a'] + 8*Level['P2'] +
   40*Level['P2a'] + 15*Level['P2b'] + 70*Level['P3'] +
   25*Level['P3c'] + 60*Level['P4'] >= 0
 complements
   0 <= Price['AA1'];
```

A predefined array of integers, \_scvar, gives the indices of the complementing variables
in the generic variable arrays \_var and \_varname. This terminology can be used
to display a list of names of such variables:

```ampl
ampl: display {i in 1..3} (_sconname[i],_svarname[_scvar[i]]);
:       _sconname[i]     _svarname[_scvar[i]]    :=
1   "Pri_Compl['AA1'].R"   "Price['AA1']"
2   "Pri_Compl['AC1'].R"   "Price['AC1']"
3   "Pri_Compl['BC1'].R"   "Price['BC1']"
;
```

When constraint `i` is an ordinary equality or inequality, `_scvar[i]` is 0. The names of
complementarity constraints in `_sconname` are suffixed with .L or .R according to
whether the expr in the constraint sent to the solver was derived from the left or right
argument to complements in the original constraint.

**Bibliography**

Richard W. Cottle, Jong-Shi Pang, and Richard E. Stone, The Linear Complementarity Problem,
Academic Press (San Diego, CA, 1992). An encyclopedic account of linear complementarity problems
with a nice overview of how these problems arise.
Steven P. Dirkse and Michael C. Ferris, "MCPLIB: A Collection of Nonlinear Mixed Complementarity
Problems." Optimization Methods and Software 5, 4 (1995) pp. 319-345. An extensive survey
of nonlinear complementarity, including problem descriptions and mathematical formulations.

Michael C. Ferris and Jong-Shi Pang, "Engineering and Economic Applications of Complementarity
Problems." SIAM Review 39, 4 (1997) pp. 669-713. A variety of complementarity test problems,
originally written in the GAMS modeling language but now in many cases translated to
AMPL.